# `AnnotationPreprocessor.encode()`

`encode` maps each annotation in `df_annot` onto the target `df_seq[sequence]` and returns a `[0, 1]`-normalized per-residue `dict_num` (`{entry: (L, D)}`) for `CPP.run_num` — one dimension per `feature_type`. The expected residue identity (`aa`) is checked at every position (`on_mismatch='raise'` by default, the off-by-isoform guard); annotated residues carry their `score`, in-coverage non-annotated residues are `0.0`.

In [1]:
import warnings
import numpy as np
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

annp = aa.AnnotationPreprocessor(verbose=False)
df_seq = pd.DataFrame({'entry': ['AF_TINY'],
                       'sequence': ['ACDEFGHIKLMNPQRSTVWYACDEFGHIKL']})
# A small user/predictor table -> Functional sites (open vocabulary).
df_user = pd.DataFrame({ut.COL_PROTEIN_ID: ['AF_TINY', 'AF_TINY'],
                        ut.COL_START: [3, 16],
                        ut.COL_FEATURE_TYPE: ['hotspot', 'hotspot'],
                        ut.COL_SCORE: [0.92, 0.40]})
df_annot = annp.ingest(df_user)

dict_num = annp.encode(df_seq=df_seq, df_annot=df_annot,
                     features=['hotspot'])
arr = dict_num['AF_TINY']
print('shape (L, D):', arr.shape)
print('annotated positions (1-based):',
      list(np.where(arr[:, 0] > 0)[0] + 1))

shape (L, D): (30, 1)
annotated positions (1-based): [np.int64(3), np.int64(16)]


In [2]:
# Further parameters: ``on_mismatch`` sets the off-by-isoform policy
# ('raise' | 'drop' | 'warn'), and ``return_df=True`` additionally returns a
# tidy long-form table of the mapped annotations.
dict_num, df_encoded = annp.encode(
    df_seq=df_seq, df_annot=df_annot, features=['hotspot'],
    on_mismatch='warn', return_df=True)
aa.display_df(df_encoded, n_rows=10, show_shape=True)

DataFrame shape: (1, 3)


,entry,sequence,encode_ok
1,AF_TINY,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,True


Stack the result with structure / embedding `dict_num`s via `aa.combine_dict_nums`, slice with `NumericalFeature.get_parts`, then run `CPP.run_num`.